In [17]:
import numpy as np
import scipy
import numpy.matlib as mtlb
from scipy.optimize import root, minimize
import matplotlib.pyplot as plt
import time
import sys
from matplotlib import cm
from matplotlib.ticker import LinearLocator

np.set_printoptions(suppress=True)
np.random.seed(100)

In [2]:
# Compute the CDF from data and x values (x values are the values at which we want to evaluate the CDF)
def cdf(err, x):
    CDF = np.zeros(len(x))
    for iter in range(0, len(x)):
        CDF[iter] = np.sum(err <= x[iter])/len(err)
    return CDF


### Function to estimate the position of the object of interest from the estimated range, angle of the reflected path and distance of the UE from the gNB
def est_pos(d, l_hat, theta_hat):
    x1 = (d + l_hat)*np.sin(theta_hat)
    y1 = (d + l_hat)*np.cos(theta_hat)
    m = (y1 - d)/x1

    x_mid = x1/2
    y_mid = (y1 + d)/2

    x_s_hat = (x_mid/m + y_mid)/(1/m + 1/np.tan(theta_hat))
    y_s_hat = x_s_hat/np.tan(theta_hat)

    return x_s_hat, y_s_hat


# Function to estimate the full channel estimates from combed channel estimates using interpolation 
def interpolate_channel_estimates(H_combed, nPRB, N_ifft, K_TC):
    H_combed = np.concatenate([H_combed[N_ifft-6*nPRB:N_ifft], H_combed[0:6*nPRB]])
    K_srs = int(np.ceil(12*nPRB / K_TC))
    H = np.zeros([12*nPRB], dtype=np.complex128)
    n = 0

    if K_TC == 2:
        filt8_start = np.array([12288, 8192, 4096, 0, 0, 0, 0, 0])/16384
        filt8_end = np.array([4096, 8192, 12288, 16384])/16384
        filt8_middle2 = np.array([4096, 8192, 8192, 8192, 4096, 0, 0, 0])/16384
        filt8_middle4 = np.array([0, 0, 4096, 8192, 8192, 8192, 4096, 0])/16384

        for k in range(0, K_srs):
            if k == 0:
                H[n:n+len(filt8_start)] = H[n:n+len(filt8_start)] + H_combed[k*K_TC]*filt8_start
            elif k == K_srs-1:
                H[n:n+len(filt8_end)] = H[n:n+len(filt8_end)] + H_combed[k*K_TC]*filt8_end
            elif np.mod(k, 2) == 1:
                H[n:n+len(filt8_middle2)] = H[n:n+len(filt8_middle2)] + H_combed[k*K_TC]*filt8_middle2
            elif np.mod(k, 2) == 0:
                H[n:n+len(filt8_middle4)] = H[n:n+len(filt8_middle4)] + H_combed[k*K_TC]*filt8_middle4
                n = k*K_TC

    elif K_TC == 4:
        filt16_start = np.array([12288, 8192, 8192, 8192, 4096, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])/16384
        filt16_end = np.array([4096, 8192, 8192, 8192, 12288, 16384, 16384, 16384, 0, 0, 0, 0, 0, 0, 0, 0])/16384
        filt16_middle = np.array([4096, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 4096, 0, 0, 0, 0, 0, 0, 0])/16384

        for k in range(0, K_srs):
            if k == 0:
                H[n:n+len(filt16_start)] = H[n:n+len(filt16_start)] + H_combed[k*K_TC]*filt16_start
            elif k == K_srs-1:
                if n+len(filt16_end) > len(H):
                    H[n:len(H)] = H[n:len(H)] + H_combed[k*K_TC]*filt16_end[0:len(H)-n]
                else:
                    H[n:n+len(filt16_end)] = H[n:n+len(filt16_end)] + H_combed[k*K_TC]*filt16_end
            else:
                if n+len(filt16_middle) > len(H):
                    H[n:len(H)] = H[n:len(H)] + H_combed[k*K_TC]*filt16_middle[0:len(H)-n]
                else:
                    H[n:n+len(filt16_middle)] = H[n:n+len(filt16_middle)] + H_combed[k*K_TC]*filt16_middle
                n = k*K_TC

    elif K_TC == 8:
        filt16_start = np.array([12288, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 4096, 0, 0, 0, 0, 0, 0, 0])/16384
        filt16_end = np.array([4096, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 12288, 16384, 16384, 16384, 16384, 16384, 16384, 16384])/16384
        filt16_middle = np.array([4096, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 8192, 4096])/16384

        for k in range(0, K_srs):
            if k == 0:
                H[n:n+len(filt16_start)] = H[n:n+len(filt16_start)] + H_combed[k*K_TC]*filt16_start
            elif k == K_srs-1:
                if n+len(filt16_end) > len(H):
                    H[n:len(H)] = H[n:len(H)] + H_combed[k*K_TC]*filt16_end[0:len(H)-n]
                else:
                    H[n:n+len(filt16_end)] = H[n:n+len(filt16_end)] + H_combed[k*K_TC]*filt16_end
            else:
                if n+len(filt16_middle) > len(H):
                    H[n:len(H)] = H[n:len(H)] + H_combed[k*K_TC]*filt16_middle[0:len(H)-n]
                else:
                    H[n:n+len(filt16_middle)] = H[n:n+len(filt16_middle)] + H_combed[k*K_TC]*filt16_middle
                n = k*K_TC
            

    H = np.concatenate([H[6*nPRB:12*nPRB], np.zeros(N_ifft - 12*nPRB), H[0:6*nPRB]])
    
    return H


# Function to find LoS path using the estimated AoA and ToF
def find_los(t_est, theta_est, alpha_est, aoa_ue):
    alpha_max = np.max(np.abs(alpha_est))
    
    while(len(t_est) > 0): # Continue until there is a path left to process
        l_ue = np.argmin(t_est) # Find the path index with the minimum transmit delay
        if (np.abs(theta_est[l_ue] - aoa_ue) > 7.5) or (np.abs(alpha_est[l_ue]) < 0.5*alpha_max): # Ignore the path if it is not coming from the UE or if it is too weak. LoS from the UE should be one of the strongest paths if not the strongest.
            t_est = np.delete(t_est, l_ue)
            theta_est = np.delete(theta_est, l_ue)
            alpha_est = np.delete(alpha_est, l_ue)
        else:
            break

    return t_est, theta_est, alpha_est, l_ue

In [3]:
### Permutations of path indices for different values of existing paths and new paths to be allocated. 
# For example, permute3_2 gives all the permutations of 2 paths allocated to 3 existing paths. 
## List of all possible permutations
permute1_3 = np.array([[0,6,6],
                       [6,0,6],
                       [6,6,0]])

permute2_3 = np.array([[0,1,6],
                      [1,0,6],
                      [0,6,1],
                      [1,6,0],
                      [6,0,1],
                      [6,1,0]])

permute3_3 = np.array([[0,1,2],
                     [0,2,1],
                     [1,0,2], 
                     [1,2,0], 
                     [2,0,1], 
                     [2,1,0]])

permute4_3 = np.array([[0,1,2],
                     [0,2,1],
                     [1,0,2], 
                     [1,2,0], 
                     [2,0,1], 
                     [2,1,0], 
                     [3,1,2],
                     [3,2,1],
                     [3,0,2], 
                     [3,2,0], 
                     [3,0,1], 
                     [3,1,0], 
                     [0,3,2],
                     [0,3,1],
                     [1,3,2], 
                     [1,3,0], 
                     [2,3,1], 
                     [2,3,0], 
                     [0,1,3],
                     [0,2,3],
                     [1,0,3], 
                     [1,2,3], 
                     [2,0,3], 
                     [2,1,3]])

permute5_3 = np.array([[0,1,2],
                     [0,2,1],
                     [1,0,2],
                     [1,2,0],
                     [2,0,1],
                     [2,1,0],
                     [3,1,2],
                     [3,2,1],
                     [3,0,2],
                     [3,2,0],
                     [3,0,1],
                     [3,1,0],
                     [0,3,2],
                     [0,3,1],
                     [1,3,2],
                     [1,3,0],
                     [2,3,1],
                     [2,3,0],
                     [0,1,3],
                     [0,2,3],
                     [1,0,3],
                     [1,2,3],
                     [2,0,3],
                     [2,1,3],
                     [4,1,2],
                     [4,2,1],
                     [4,0,2],
                     [4,2,0],
                     [4,0,1],
                     [4,1,0],
                     [0,4,2],
                     [0,4,1],
                     [1,4,2],
                     [1,4,0],
                     [2,4,1],
                     [2,4,0],
                     [0,1,4],
                     [0,2,4],
                     [1,0,4],
                     [1,2,4],
                     [2,0,4],
                     [2,1,4],
                     [4,1,3],
                     [4,2,3],
                     [4,0,3],
                     [4,3,0],
                     [4,3,1],
                     [4,3,2],
                     [0,4,3],
                     [1,4,3],
                     [2,4,3],
                     [3,4,0],
                     [3,4,1],
                     [3,4,2],
                     [0,3,4],
                     [1,3,4],
                     [2,3,4],
                     [3,0,4],
                     [3,1,4],
                     [3,2,4]])

permute1_2 = np.array([[0,6],
                       [6,0]])

permute2_2 = np.array([[0,1],
                       [1,0]])

permute3_2 = np.array([[0,1],
                       [0,2],
                       [1,0],
                       [1,2],
                       [2,0],
                       [2,1]])

permute4_2 = np.array([[0,1],
                       [0,2],
                       [0,3],
                       [1,0],
                       [1,2],
                       [1,3],
                       [2,0],
                       [2,1],
                       [2,3],
                       [3,0],
                       [3,1],
                       [3,2]])

permute5_2 = np.array([[0,1],
                      [0,2],
                      [0,3],
                      [0,4],
                      [1,0],
                      [1,2],
                      [1,3],
                      [1,4],
                      [2,0],
                      [2,1],
                      [2,3],
                      [2,4],
                      [3,0],
                      [3,1],
                      [3,2],
                      [3,4],
                      [4,0],
                      [4,1],
                      [4,2],
                      [4,3]])

In [4]:
### Signal and system parameters
D = 11 # distance of the UE from the gNB (in m)
c = 299792458
f_c = 3.6192e9
bandwidth = 20e6
scs = 30e3
K_TC = 2 # This is the SRS transmission comb

if bandwidth == 20e6:
    f_s = 30.72e6
    N_ifft = 1024
    nPRB = 51
    N_cp = 72
    occ_bandwidth = 12*nPRB*scs
    print("Bandwidth is 20 MHz")

elif bandwidth == 40e6:
    f_s = 61.44e6
    N_ifft = 2048
    nPRB = 106
    N_cp = 144
    occ_bandwidth = 12*nPRB*scs
    print("Bandwidth is 40 MHz")

elif bandwidth == 80e6:
    f_s = 122.88e6
    N_ifft = 4096
    nPRB = 217
    N_cp = 288
    occ_bandwidth = 12*nPRB*scs
    print("Bandwidth is 80 MHz")

osf = 8
N_ant = 4
d_ant = 1/2*np.arange(0, N_ant, 1) # inter-element distance (in terms of wavelength)
P_t = 0.1 # Watt
K = 1.380649e-23
T = 298
noise_var = K*bandwidth*T


### Channel parameters
chan_params = np.load('/home/bkumar2/Documents/Research/ISAC-twin/Simulation_framework/Datasets/Indoor_scenario/Trajectory_1/ISAC_ray_tracing_chan.npy')
positions = np.load('/home/bkumar2/Documents/Research/ISAC-twin/Simulation_framework/Datasets/Indoor_scenario/Trajectory_1/ISAC_locations.npy')
obj_params = np.load('/home/bkumar2/Documents/Research/ISAC-twin/Simulation_framework/Datasets/Indoor_scenario/Trajectory_1/obj_params.npy')

K = 100
tau = chan_params[:, 2, :] # Transmit delay in s 
theta = chan_params[:, 0, :] # AoA in radians
Doppler = chan_params[:, 1, :] # Doppler in Hz (converted from m/s)
pathgain = chan_params[:, 3, :] # Pathgain in absolute value

for k in range(0, K):
    order = np.argsort(pathgain[k, :])[::-1]
    tau[k, :] = tau[k, order]
    theta[k, :] = theta[k, order]
    Doppler[k, :] = Doppler[k, order]
    pathgain[k, :] = pathgain[k, order]

# Repeat the same values K times for averaging (Monte carlo simulation)
# tau = np.tile(np.reshape(tau, [1, len(tau)]), [K, 1])
# theta = np.tile(np.reshape(theta, [1, len(theta)]), [K, 1])
theta = theta + np.pi
theta = np.pi/2 - theta
theta = (theta + np.pi) % (2 * np.pi) - np.pi
obj_params[:, 0] = np.pi/180*obj_params[:, 0] 
obj_params[:, 0] = obj_params[:, 0] + np.pi
obj_params[:, 0] = np.pi/2 - obj_params[:, 0]
obj_params[:, 0] = (obj_params[:, 0] + np.pi) % (2 * np.pi) - np.pi
obj_params[:, 0] = 180/np.pi*obj_params[:, 0]
# Doppler = np.tile(np.reshape(Doppler, [1, len(Doppler)]), [K, 1])
pathgain = P_t*pathgain/noise_var

Bandwidth is 20 MHz


In [6]:
n_loop = 1
err = np.array([])
err_aoa = np.array([])
err_delay = np.array([])

time1 = time.time() 

# Generate the received signal

# No. of paths
L = np.shape(tau)[1]

# Generated signal for K scenarios, N_ant antennas and N_ifft samples
gen_signal = np.zeros([K, N_ant, N_ifft], dtype=np.complex128)

# Transmitted OFDM symbols
# Xn = np.random.randint(0, 4, [12*nPRB]) # QPSK symbols
# Xn = np.exp(1j*(np.pi*Xn/2 + np.pi/4))

Xn = np.zeros([12*nPRB], dtype=np.complex128)
m = np.arange(0, int(np.ceil(12*nPRB / K_TC)), 1)
Xn[np.arange(0, 12*nPRB, K_TC)] = np.exp(-1j*np.pi*m*(m + 2)/(len(m))) # SRS pilot symbols (ZC sequence)
Xn[np.setdiff1d(np.arange(0, 12*nPRB, 1), np.arange(0, 12*nPRB, K_TC))] = np.exp(1j*(np.pi*np.random.randint(0, 4, [12*nPRB - len(m)])/2 + np.pi/4)) # Remaining subcarriers transmit QPSK symbols
Xn = np.concatenate([Xn[6*nPRB:12*nPRB], np.zeros(N_ifft - 12*nPRB), Xn[0:6*nPRB]])
Xn = np.reshape(Xn, [N_ifft, 1])

X_pilot = np.zeros([12*nPRB], dtype=np.complex128)
X_pilot[np.arange(0, 12*nPRB, K_TC)] = np.exp(-1j*np.pi*m*(m + 2)/(len(m)))
X_pilot = np.concatenate([X_pilot[6*nPRB:12*nPRB], np.zeros(N_ifft - 12*nPRB), X_pilot[0:6*nPRB]])

t = np.arange(N_cp, N_cp+N_ifft, 1)/f_s # Sampling timestamps, starts from N_cp because we discard the first N_cp samples
n = np.concatenate([np.arange(0, N_ifft//2, 1), np.arange(-N_ifft//2, 0, 1)])
n = np.reshape(n, [N_ifft, 1])
n = np.tile(n, [1, N_ifft])

# This loop just generates the OFDM samples in time domain
for k in range(0, K):
    for l in range(0, L):
        if (theta[k, l] >= -np.pi/2) and (theta[k, l] < np.pi/2):
            phase = np.exp(1j*2*np.pi*np.random.rand())
            temp_sgnl = 1/(np.sqrt(N_ifft))*np.sqrt(pathgain[k, l]) * phase * \
                        np.exp(1j*2*np.pi*Doppler[k, l]*t) * \
                        np.sum(np.tile(Xn, [1, N_ifft])*np.exp(1j*2*np.pi*scs*n*np.tile(np.reshape(t-N_cp/f_s-tau[k, l], [1, N_ifft]), [N_ifft, 1])), axis=0)

            gen_signal[k, :, :] = gen_signal[k, :, :] + np.tile(np.reshape(np.exp(1j*2*np.pi*d_ant*np.sin(theta[k, l])), [N_ant, 1]), [1, N_ifft]) * \
                                                        np.tile(np.reshape(temp_sgnl, [1, N_ifft]), [N_ant, 1])

# Add the AWGN noise to the generated samples
rcvd_signal = gen_signal #+ 1/np.sqrt(2)*(np.random.normal(size=[K, N_ant, N_ifft]) + 1j*np.random.normal(size=[K, N_ant, N_ifft]))

# Convert the received samples back to frequency domain to estimate the OFDM symbols or channel
rcvd_sig_freq = 1/np.sqrt(N_ifft)*np.fft.fft(rcvd_signal, axis=2)

# Estimate the channel using X_pilot as pilot symbols
H_combed = np.zeros([K, N_ant, N_ifft], dtype=np.complex128) 

for i in range(0, N_ifft):
    if X_pilot[i] == 0:
        H_combed[:, :, i] = np.zeros([K, N_ant])
    else:
        H_combed[:, :, i] = rcvd_sig_freq[:, :, i]*np.conjugate(np.tile(X_pilot[i], [K, N_ant]))

H = np.zeros([K, N_ant, N_ifft], dtype=np.complex128)
for k in range(0, K):
    for n_ant in range(0, N_ant):
        H[k, n_ant, :] = interpolate_channel_estimates(H_combed[k, n_ant, :], nPRB, N_ifft, K_TC)


L = 3 # No. of paths we are estimating

track_params = np.zeros([10, 5, 2])
track_idx = 0
n_paths = np.zeros(np.shape(track_params)[1])

m = np.arange(0, 12*nPRB, 1) # New and longer set of SRS symbols (ZC sequence)
Xn = np.exp(-1j*np.pi*m*(m + 2)/(len(m)))
Xn = np.concatenate([Xn[6*nPRB:12*nPRB], np.zeros(N_ifft - 12*nPRB), Xn[0:6*nPRB]])
Xn = np.reshape(Xn, [N_ifft, 1])

# First step is regenerating the received samples in time domain from the estimated channel and known pilot symbols (Xn) for all 4 different antennas
complex_rxsym0 = H[:, 0, :]*np.tile(np.reshape(Xn[:, 0], [1, N_ifft]), [K, 1])
complex_rxsym0 = np.concatenate([complex_rxsym0[:, 0:N_ifft//2], np.zeros([K, (osf-1)*N_ifft]), complex_rxsym0[:, N_ifft//2:N_ifft]], axis=1)
complex_time0 = np.sqrt(osf*N_ifft)*np.fft.ifft(complex_rxsym0, axis=1)
complex_rxsym1 = H[:, 1, :]*np.tile(np.reshape(Xn[:, 0], [1, N_ifft]), [K, 1])
complex_rxsym1 = np.concatenate([complex_rxsym1[:, 0:N_ifft//2], np.zeros([K, (osf-1)*N_ifft]), complex_rxsym1[:, N_ifft//2:N_ifft]], axis=1)
complex_time1 = np.sqrt(osf*N_ifft)*np.fft.ifft(complex_rxsym1, axis=1)
complex_rxsym2 = H[:, 2, :]*np.tile(np.reshape(Xn[:, 0], [1, N_ifft]), [K, 1])
complex_rxsym2 = np.concatenate([complex_rxsym2[:, 0:N_ifft//2], np.zeros([K, (osf-1)*N_ifft]), complex_rxsym2[:, N_ifft//2:N_ifft]], axis=1)
complex_time2 = np.sqrt(osf*N_ifft)*np.fft.ifft(complex_rxsym2, axis=1)
complex_rxsym3 = H[:, 3, :]*np.tile(np.reshape(Xn[:, 0], [1, N_ifft]), [K, 1])
complex_rxsym3 = np.concatenate([complex_rxsym3[:, 0:N_ifft//2], np.zeros([K, (osf-1)*N_ifft]), complex_rxsym3[:, N_ifft//2:N_ifft]], axis=1)
complex_time3 = np.sqrt(osf*N_ifft)*np.fft.ifft(complex_rxsym3, axis=1)

# This is the IFFT samples of the OFDM symbols (This is later used in estimating the Doppler and ToA using Ambiguity function)
Xn_os = np.concatenate([Xn[0:N_ifft//2, 0], np.zeros((osf-1)*N_ifft), Xn[N_ifft//2:N_ifft, 0]])
time_gensym = np.sqrt(osf*N_ifft)*np.fft.ifft(Xn_os)

In [37]:
%matplotlib qt

t_search = np.arange(-50, 50, 1)
theta_search = np.arange(-5*np.pi/12, 5*np.pi/12, 0.1)
fft_2d = np.zeros([K, len(theta_search), len(t_search)])

for iter0 in range(0, 35):
    est_mean_time0 = complex_time0[iter0, :]
    est_mean_time1 = complex_time1[iter0, :]
    est_mean_time2 = complex_time2[iter0, :]
    est_mean_time3 = complex_time3[iter0, :]

    mean_chan_subcarrier = np.concatenate([[est_mean_time0], [est_mean_time1], [est_mean_time2], [est_mean_time3]], axis=0)

    print("-------------------- Symbol no. {} --------------------".format(iter0))

    for iter_t in range(0, len(t_search)):
        for iter_theta in range(0, len(theta_search)):
            # n = np.concatenate([np.arange(0, N_ifft//2, 1), np.arange(-N_ifft//2, 0, 1)])/N_ifft
            # delay_multiplier = np.tile(np.reshape(np.exp(1j*2*np.pi*n*t_search[iter_t]/osf), [1, N_ifft]), [N_ant, 1])
            # angle_multiplier = np.tile(np.reshape(np.exp(-1j*2*np.pi*d_ant*np.sin(theta_search[iter_theta])), [N_ant, 1]), [1, N_ifft])
            # fft_multiplier = angle_multiplier*delay_multiplier
            
            # fft_2d[iter_theta, iter_t] = np.abs(np.sum(mean_chan_subcarrier*fft_multiplier))

            combined_chan = np.sum(np.tile(np.reshape(np.exp(-1j*2*np.pi*d_ant*np.sin(theta_search[iter_theta])), [N_ant, 1]), [1, osf*N_ifft])*mean_chan_subcarrier, axis=0)
            fft_2d[iter0, iter_theta, iter_t] = np.abs(np.sum(np.roll(np.conj(time_gensym), t_search[iter_t])*combined_chan))

    fft_2d[iter0, :, :] = fft_2d[iter0, :, :]/np.max(fft_2d[iter0, :, :])
    # fft_2d = 10*np.log10(fft_2d)
    # plt.figure(iter0)
    # plt.figure(figsize=(20, 50))
    # plt.imshow(fft_2d)
    # plt.colorbar()
    # plt.yticks(ticks=np.arange(0, len(theta_search), 5), labels=np.round(180/np.pi*theta_search[np.arange(0, len(theta_search), 5)], 0))
    # plt.xticks(ticks=np.arange(0, len(t_search), 5), labels=np.round(1e9/(osf*f_s)*t_search[np.arange(0, len(t_search), 5)], 0))
    
    # fig = plt.figure(figsize =(14, 9))
    # ax = plt.axes(projection ='3d')
    # X, Y = np.meshgrid(theta_search, t_search)
    # ax.plot_surface(X, Y, fft_2d)
#     plt.ylabel("AoA in degrees", fontsize=30)
#     plt.xlabel("Transmit delay in ns", fontsize=30)

# plt.show()

-------------------- Symbol no. 0 --------------------
-------------------- Symbol no. 1 --------------------
-------------------- Symbol no. 2 --------------------
-------------------- Symbol no. 3 --------------------
-------------------- Symbol no. 4 --------------------
-------------------- Symbol no. 5 --------------------
-------------------- Symbol no. 6 --------------------
-------------------- Symbol no. 7 --------------------
-------------------- Symbol no. 8 --------------------
-------------------- Symbol no. 9 --------------------
-------------------- Symbol no. 10 --------------------
-------------------- Symbol no. 11 --------------------
-------------------- Symbol no. 12 --------------------
-------------------- Symbol no. 13 --------------------
-------------------- Symbol no. 14 --------------------
-------------------- Symbol no. 15 --------------------
-------------------- Symbol no. 16 --------------------
-------------------- Symbol no. 17 --------------------
--

In [43]:
diff = fft_2d[32, :, :] - fft_2d[0, :, :]

np.unravel_index(np.argmax(diff), diff.shape)

(np.int64(7), np.int64(62))

In [41]:

fig, ax = plt.subplots(subplot_kw={"projection": "3d"})

# Make data.
X, Y = np.meshgrid(t_search, theta_search)
Z = diff

# Plot the surface.
surf = ax.plot_surface(X, Y, Z, cmap=cm.coolwarm,
                       linewidth=0, antialiased=False)

# Customize the z axis.
ax.set_zlim(-1.01, 1.01)
ax.zaxis.set_major_locator(LinearLocator(10))
# A StrMethodFormatter is used automatically
ax.zaxis.set_major_formatter('{x:.02f}')

# Add a color bar which maps values to colors.
fig.colorbar(surf, shrink=0.5, aspect=5)

In [22]:
np.shape(fft_2d)

(100, 27)

In [32]:
idx = 30

print(tau[idx, :]*1e9)
print(theta[idx, :]*180/np.pi)
print(pathgain[idx, :])

[ 36.69205029  37.26728792  52.67728653  63.0304271   46.69813336
  46.69813336  81.43297237  50.99102651  49.67253275  49.67253275
  37.29359577  46.79024102 103.17806698  67.01459654 106.93750596
  57.21059892  57.21059182   0.           0.           0.
   0.           0.           0.           0.           0.        ]
[   0.0000025     0.0000025     2.29657848    0.0000025     0.0000025
    0.0000025  -164.3862447     0.0000025     6.902619     -6.90261399
    0.0000025     0.0000025     2.15106813 -170.17214576   -7.40644973
  141.57514956 -141.57512384  -90.          -90.          -90.
  -90.          -90.          -90.          -90.          -90.        ]
[466627.72255199  61145.71815245  43843.02391641  14703.65780194
   9619.24915765   9619.07753769   1410.98808008    683.68764622
    616.34286721    616.34151799    369.8291764     113.87527464
     14.7704097      11.92822711     11.51051903      1.17852326
      1.17838412      0.              0.              0.
      0.     